# A-Network — 物理神经网络训练

基于 PyTorch 的 3D 张量场物理神经网络。
Token 注入 80×80×80 场中，经 20 步 26-邻域扩散传播后读出为 logits。

**仓库**: https://github.com/Mn3TR/a-network

**功能**:
- 支持 GPU 加速（Colab T4/V100）
- 支持 batch 训练（多序列并行）
- 支持在线数据集（HuggingFace / URL）

## 1. 环境准备

In [ ]:
import os
# 克隆仓库
!git clone https://github.com/Mn3TR/a-network.git
%cd a-network

In [ ]:
# 安装依赖
!pip install -q torch tokenizers numpy matplotlib datasets

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. 挂载 Google Drive（可选）

挂载后权重和日志会持久保存，Colab 断开也不会丢失。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/a-network/output /content/drive/MyDrive/a-network/log
print('Drive ready')

## 3. 配置训练参数

可通过环境变量覆盖配置：

In [ ]:
# ========== 训练参数 ==========
TRAIN_EPOCHS = 5          # 训练轮数
LEARNING_RATE = 1e-4
GRAD_ACCUM = 4

# ========== Batch 大小 ==========
# batch_size > 1 时，多个独立序列并行训练
# 每个序列有自己的场状态，共享网络参数
# 增大 batch_size 会使训练更快，但每步内存更大
BATCH_SIZE = 1             # 设为 2/4/8 启用 batch 训练

# ========== 数据集 ==========
# HuggingFace 数据集名，例如:
#   "roneneldan/TinyStories"
#   "roneneldan/TinyStories,split=train"
DATASET = "roneneldan/TinyStories"

# ========== 持久化 ==========
USE_DRIVE = False         # 挂载 Drive 后改为 True

In [ ]:
import sys
sys.path.insert(0, '.')

from src.anetwork.config import ANetworkConfig, TrainConfig
from src.anetwork.model import ANetwork
from src.anetwork.tokenizer import TokenizerWrapper

# 设置环境变量（train.py 会读取它们）
os.environ['DATASET'] = f'huggingface:{DATASET}'
os.environ['BATCH_SIZE'] = str(BATCH_SIZE)

# 加载 tokenizer
tokenizer = TokenizerWrapper('tokenizer/tokenizer.json')
print(f'Vocab size: {tokenizer.vocab_size}')

# 创建网络
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
a_cfg = ANetworkConfig(vocab_size=tokenizer.vocab_size)
net = ANetwork(a_cfg, device=DEVICE).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in net.parameters()):,}')

## 4. 加载数据

从 HuggingFace Datasets 在线加载。

In [ ]:
from src.anetwork.data import load_data, pack_batch, count_batch_steps

tokens = load_data(DATASET, tokenizer)
total_steps = count_batch_steps(tokens, BATCH_SIZE)
print(f'Tokens: {len(tokens)}, Batch steps/epoch: {total_steps}')

# 预览 batch 数据
print(f'\nBatch samples (B={BATCH_SIZE}):')
for i, (inp, tgt) in enumerate(pack_batch(tokens, BATCH_SIZE)):
    if i >= 3: break
    print(f'  step {i}: in={inp}, tgt={tgt}')

## 5. 开始训练

每个 epoch 结束后自动保存权重。
场快照保存在 `log/` 目录，可用下方可视化查看。

In [ ]:
import math, time
from pathlib import Path

B = BATCH_SIZE
N = a_cfg.N
output_dir = '/content/drive/MyDrive/a-network' if USE_DRIVE else '.'

optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)

for epoch in range(TRAIN_EPOCHS):
    # 余弦学习率退火
    if TRAIN_EPOCHS > 1:
        p = epoch / (TRAIN_EPOCHS - 1)
        factor = (1.0 + math.cos(math.pi * p)) * 0.5
        lr = 1e-6 + (LEARNING_RATE - 1e-6) * factor
        for pg in optimizer.param_groups:
            pg['lr'] = lr

    net.train()
    epoch_loss = 0.0
    epoch_start = time.time()
    optimizer.zero_grad()

    # 初始化场状态
    if B > 1:
        fields = torch.zeros(B, N, device=DEVICE)
        incomings = torch.zeros(B, N, device=DEVICE)
    else:
        net.reset_state()

    for step_idx, (inputs, targets) in enumerate(pack_batch(tokens, B)):
        input_ids = torch.tensor(inputs, device=DEVICE)
        target_ids = torch.tensor(targets, device=DEVICE)

        if B > 1:
            loss, fields, incomings = net.train_step_batch(
                fields, incomings, input_ids, target_ids)
        else:
            loss = net.train_step(input_ids[0], target_ids[0])

        loss.backward()
        epoch_loss += loss.item()

        if (step_idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        if (step_idx + 1) % max(1, total_steps // 10) == 0:
            print(f'  e{epoch} [{step_idx+1}/{total_steps}] loss={loss.item():.4f}')

    if total_steps % GRAD_ACCUM != 0:
        optimizer.step()
        optimizer.zero_grad()

    epoch_s = time.time() - epoch_start
    avg_loss = epoch_loss / total_steps
    print(f'Epoch {epoch}: avg_loss={avg_loss:.4f}  time={epoch_s:.1f}s  lr={lr:.8f}')

    # 保存
    net.save(f'{output_dir}/output/weights_epoch{epoch}.pt')

print('Training done!')

## 6. 生成文本

用训练好的模型自回归生成文本。

In [ ]:
net.eval()
seed_text = 'Time'
seed_ids = tokenizer.encode(seed_text)[:3]
print(f'Seed: "{seed_text}"')

generated = net.generate(seed_ids, max_tokens=50)
output = tokenizer.decode(generated)
print(f'Output: "{output}"')

## 7. 场状态可视化

查看 80×80×80 场的三切片。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

field = net.field.detach().cpu().numpy().reshape(80, 80, 80)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for i, z in enumerate([20, 40, 60]):
    im = axes[i].imshow(field[:, :, z], cmap='viridis')
    axes[i].set_title(f'z={z}')
    axes[i].axis('off')
fig.colorbar(im, ax=axes, shrink=0.6)
plt.show()